In [ ]:
import os
from glob import glob
from pathlib import Path
from random import shuffle

import ipyplot
import torch
from lightning.pytorch import Trainer
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset

import notebook_setup  # isort:skip
from acid import conf  # isort:skip
from acid.torch.setup import SquareModelSetup  # isort:skip
from acid.torch.lightning_modules import SquareClassifierModule  # isort:skip
from acid.helper import order_points, four_point_transform  # isort:skip

In [ ]:
BATCH_SIZE = 128

In [ ]:
image_paths = list(SquareModelSetup.dataset_path.rglob("*.jp*"))


class TestDataset(Dataset):
    def __init__(self, transform=False):
        self.transform = transform
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_filepath = self.image_paths[idx]
        image = Image.open(image_filepath)
        if self.transform is not None:
            image = self.transform(image)
        return image


dataset = TestDataset(transform=SquareModelSetup.transforms["val"])
model = SquareModelSetup.model
trainer = Trainer(accelerator="auto", default_root_dir=conf.NOTEBOOKS_TMP_DIR)

In [ ]:
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)
predictions = trainer.predict(model, dataloader)

In [ ]:
results = []
for batch in predictions:
    prob = nn.functional.softmax(batch, dim=1)
    pred_probabilities, pred_classes = prob.topk(1, dim=1)
    for idx, pred_probability in enumerate(pred_probabilities):
        results.append((float(pred_probability), int(pred_classes[idx])))

In [ ]:
# Plot expected class <> predicted class

loss_images, loss_labels = [], []
for idx, image_path in enumerate(image_paths):
    # training/data/squares/sortme/empty/f0/f0/foo.jpeg => empty
    class_name_expected = image_path.parent.parent.name
    class_id_expected = SquareModelSetup.classes.index(class_name_expected)
    class_id_predicted = results[idx][1]
    class_name_predicted = SquareModelSetup.classes[class_id_predicted]
    probability = int(results[idx][0] * 100)

    if class_id_predicted != class_id_expected:
        loss_images.append(str(image_path))
        loss_labels.append(f"exp. {class_name_expected} vs {class_name_predicted}")

print(f"Expected class <> Predicted class - ({len(loss_images)})")
ipyplot.plot_class_tabs(loss_images, loss_labels, img_width=125)

In [ ]:
# Plot top losses

loss_images, loss_labels = [], []
for idx, image_path in enumerate(image_paths):
    # training/data/squares/sortme/empty/f0/f0/foo.jpeg => empty
    class_name_expected = image_path.parent.parent.name
    class_id_expected = SquareModelSetup.classes.index(class_name_expected)
    class_id_predicted = results[idx][1]
    class_name_predicted = SquareModelSetup.classes[class_id_predicted]
    probability = int(results[idx][0] * 100)

    if probability < 30:
        loss_label = "<30"
    elif probability < 85:
        loss_label = "<85"
    elif probability < 90:
        loss_label = "<90"
    elif probability < 95:
        loss_label = "<95"
    else:
        continue

    loss_images.append(str(image_path))
    loss_labels.append(f"exp. {class_name_expected} {loss_label}")

print(f"Top losses - ({len(loss_images)})")
ipyplot.plot_class_tabs(loss_images, loss_labels, img_width=125)